In [1]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import re
import numpy as np
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
ITAD_KEY = os.getenv('ITAD_API_KEY')
DB_PATH  = os.getenv('STRING_DB_PATH')

# Funciones a usar

In [4]:
def Query(query):
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df 

# obtener dataframes

In [37]:
df_Hist_Precios_ITAD = Query("SELECT * FROM Hist_Precios_ITAD")
df_cat_tienda = Query("SELECT * FROM CAT_Tienda")

df_datos_actuales_itad = Query("SELECT * FROM Datos_Actuales_ITAD")

In [53]:
df_cat_juegos = Query("""
    SELECT j.*
    FROM CAT_Juego j
    WHERE j.juego_id NOT IN (SELECT juego_id_dlc FROM REL_Juego_DLC)
""")

In [39]:
df_rel_juego_itad = Query("SELECT * FROM REL_Juego_ITAD")

In [51]:
df_generos = Query("""
    SELECT r.juego_id, GROUP_CONCAT(g.nombre, '|') AS generos
    FROM REL_Juego_Genero r
    JOIN CAT_Genero g ON r.genero_id = g.genero_id
    GROUP BY r.juego_id
                   
""")

In [ ]:
df_modos = Query("""
    SELECT r.juego_id, GROUP_CONCAT(m.nombre, '|') AS modos
    FROM REL_Juego_Modo r
    JOIN CAT_Modo_Juego m ON r.modo_id = m.modo_id
                 WHERE j.juego_id NOT IN (SELECT juego_id_dlc FROM REL_Juego_DLC)
    GROUP BY r.juego_id
""")
#df_modos

In [44]:
df_perspectivas = Query("""
    SELECT r.juego_id, GROUP_CONCAT(p.nombre, '|') AS perspectivas
    FROM REL_Juego_Perspectiva r
    JOIN CAT_Perspectiva p ON r.perspectiva_id = p.perspectiva_id
    GROUP BY r.juego_id
""")

In [47]:
df_resenas_agg = Query("""
    SELECT
        juego_id,
        COUNT(*) AS total_resenas,
        ROUND(100.0 * SUM(recomendado) / COUNT(*), 2) AS pct_positivo,
        ROUND(AVG(minutos_al_resenar), 0) AS avg_minutos_resena,
        ROUND(AVG(puntuacion_ponderada), 4) AS avg_score_ponderado
    FROM Hist_Steam_Reviews
    GROUP BY juego_id
""")
#df_resenas_agg

In [54]:
print(f"df_Hist_Precios_ITAD:{df_Hist_Precios_ITAD.shape}")
print(f"df_cat_tienda:{df_cat_tienda.shape}")
print(f"df_datos_actuales_itad:{df_datos_actuales_itad.shape}")
print(f"df_cat_juegoss:{df_cat_juegos.shape}")
print(f"df_rel_juego_itad:{df_rel_juego_itad.shape}")
print(f"df_generos:{df_generos.shape}")
print(f"df_modos:{df_modos.shape}")
print(f"df_perspectivas:{df_perspectivas.shape}")
print(f"df_resenas_agg:{df_resenas_agg.shape}")

df_Hist_Precios_ITAD:(548468, 5)
df_cat_tienda:(30, 2)
df_datos_actuales_itad:(6077, 8)
df_cat_juegoss:(8797, 24)
df_rel_juego_itad:(6545, 2)
df_generos:(10125, 2)
df_modos:(10066, 2)
df_perspectivas:(9352, 2)
df_resenas_agg:(5205, 5)


# Desarrollo


In [36]:
#no todas la tiendas tienen datos, lo mejor sera quitar essos outliers con pocos registros
df_Hist_Precios_ITAD['id_tienda'].value_counts()

id_tienda
61    71612
6     54498
35    52772
20    49576
68    29151
37    26320
24    24865
29    21140
2     20574
15    20390
26    19336
28    19277
27    19104
64    17937
42    17354
49    15543
13    13009
66    11040
73    10929
50    10739
16    10717
47     3729
48     3681
62     1603
25     1500
52      918
72      694
75      211
4       177
17       72
Name: count, dtype: int64

In [35]:
df_cat_tienda

,id_tienda,nombre
0,2,AllYouPlay
1,4,Blizzard
2,6,Fanatical
3,13,DLGamer
4,15,Dreamgame
5,16,Epic Game Store
6,17,FireFlower
7,20,GameBillet
8,24,GamersGate
9,25,Gamesload


In [89]:
UMBRAL_TIENDA = 10_000

tiendas_validas = (df_Hist_Precios_ITAD
                   .groupby('id_tienda')
                   .size()
                   .reset_index(name='registros')
                   .query('registros >= @UMBRAL_TIENDA')
                   ['id_tienda']
                   .tolist())
tiendas_validas

[2,
 6,
 13,
 15,
 16,
 20,
 24,
 26,
 27,
 28,
 29,
 35,
 37,
 42,
 49,
 50,
 61,
 64,
 66,
 68,
 73]

In [105]:
df_df_Hist_Precios_ITAD_limpio = df_Hist_Precios_ITAD[
    df_Hist_Precios_ITAD['id_tienda'].isin(tiendas_validas)
].copy()
#df_df_Hist_Precios_ITAD_limpio
df_df_Hist_Precios_ITAD_limpio['id_tienda'].value_counts()

id_tienda
61    71612
6     54498
35    52772
20    49576
68    29151
37    26320
24    24865
29    21140
2     20574
15    20390
26    19336
28    19277
27    19104
64    17937
42    17354
49    15543
13    13009
66    11040
73    10929
50    10739
16    10717
Name: count, dtype: int64

In [94]:
print(f"{len(df_Hist_Precios_ITAD):,}")
print(f"{len(df_df_Hist_Precios_ITAD_limpio):,}")
print(f"{len(df_Hist_Precios_ITAD) - len(df_df_Hist_Precios_ITAD_limpio):,}")


548,468
535,883
12,585


In [96]:
print("Tiendas que quedaron:")
print(df_df_Hist_Precios_ITAD_limpio
      .groupby('id_tienda')
      .size()
      .reset_index(name='registros')
      .merge(df_cat_tienda, on='id_tienda')
      .sort_values('registros', ascending=False)
      .to_string(index=False))

Tiendas que quedaron:
 id_tienda  registros          nombre
        61      71612           Steam
         6      54498       Fanatical
        35      52772             GOG
        20      49576      GameBillet
        68      29151      Gamer Thor
        37      26320    Humble Store
        24      24865      GamersGate
        29      21140  GamesPlanet US
         2      20574      AllYouPlay
        15      20390       Dreamgame
        26      19336  GamesPlanet UK
        28      19277  GamesPlanet FR
        27      19104  GamesPlanet DE
        64      17937    WinGameStore
        42      17354 IndieGala Store
        49      15543          Newegg
        13      13009         DLGamer
        66      11040          Noctre
        73      10929      PlanetPlay
        50      10739          Nuuvem
        16      10717 Epic Game Store


In [97]:
df_Hist_Precios_ITAD=df_df_Hist_Precios_ITAD_limpio.copy()

In [98]:
df_Hist_Precios_ITAD['fecha'] = pd.to_datetime(df_Hist_Precios_ITAD['fecha_unix'], unit='s')
df_Hist_Precios_ITAD = df_Hist_Precios_ITAD.sort_values(['itad_id_texto', 'id_tienda', 'fecha'])
df_Hist_Precios_ITAD

,itad_id_texto,id_tienda,precio,corte_descuento,fecha_unix,fecha
362157,018d937e-e9ab-70f4-bd05-1db79cec46d2,16,29.99,0,1749505064,2025-06-09 21:37:44
362126,018d937e-e9ab-70f4-bd05-1db79cec46d2,16,5.99,80,1765572899,2025-12-12 20:54:59
362120,018d937e-e9ab-70f4-bd05-1db79cec46d2,16,29.99,0,1767891803,2026-01-08 17:03:23
362112,018d937e-e9ab-70f4-bd05-1db79cec46d2,16,5.99,80,1771408091,2026-02-18 09:48:11
362107,018d937e-e9ab-70f4-bd05-1db79cec46d2,16,29.99,0,1772185683,2026-02-27 09:48:03
...,...,...,...,...,...,...
197022,019d2aa9-203c-7306-a681-237e427bc869,61,47.99,20,1766088734,2025-12-18 20:12:14
197021,019d2aa9-203c-7306-a681-237e427bc869,61,59.99,0,1767642080,2026-01-05 19:41:20
197020,019d2aa9-203c-7306-a681-237e427bc869,61,44.99,25,1770315878,2026-02-05 18:24:38
197019,019d2aa9-203c-7306-a681-237e427bc869,61,59.99,0,1771525279,2026-02-19 18:21:19


In [109]:
desc_dist = (df_Hist_Precios_ITAD[df_Hist_Precios_ITAD['corte_descuento'] > 0]['corte_descuento']
             .value_counts()
             .sort_index()
             )
#display(desc_dist.head(20))
histograma = px.bar(desc_dist)
histograma.show()

# los descuentos no son valores continuos, son categoricos y se ve cuales sonm lo preferidos
#decuentos de 10, 80,75, 90,50,60,85,70.17,15,20

In [100]:
#revisar como estan balanecadas esas opcioens de descuento
for umbral in [10,	15	,20	,50	,60	,70	,75,	80	,85,	90]:
    elementos = (df_Hist_Precios_ITAD['corte_descuento'] >= umbral).sum()
    total = len(df_Hist_Precios_ITAD)
    print(f"descuento de {umbral}%: {elementos:,} elementos, {100*elementos/total:.1f}%")

#ser multi clase sera la mejor opcion

descuento de 10%: 325,328 elementos, 60.7%
descuento de 15%: 277,679 elementos, 51.8%
descuento de 20%: 259,146 elementos, 48.4%
descuento de 50%: 215,628 elementos, 40.2%
descuento de 60%: 187,303 elementos, 35.0%
descuento de 70%: 154,770 elementos, 28.9%
descuento de 75%: 136,201 elementos, 25.4%
descuento de 80%: 94,630 elementos, 17.7%
descuento de 85%: 51,121 elementos, 9.5%
descuento de 90%: 29,645 elementos, 5.5%


In [103]:
# combinando juegos y tiuenda se tienen muy pocos elementos por tienda como para poder usar lstm o conluciones de 1D
registros_por_par = df_Hist_Precios_ITAD.groupby(['itad_id_texto', 'id_tienda']).size()
#display(registros_por_par.head(30))
display(registros_por_par.describe())


count    36041.000000
mean        14.868705
std         32.281792
min          1.000000
25%          5.000000
50%         13.000000
75%         19.000000
max       2341.000000
dtype: float64

In [102]:
rangos = pd.cut(registros_por_par, 
                bins=[0, 4, 9, 19, 49, 99, float('inf')],
                labels=['1-4', '5-9', '10-19', '20-49', '50-99', '100+'])
rangos.value_counts().sort_index()

1-4       7529
5-9       6467
10-19    13683
20-49     7724
50-99      449
100+       189
Name: count, dtype: int64

In [110]:
#registros por tienda
por_tienda = (df_Hist_Precios_ITAD.groupby('id_tienda')
              .size()
              .reset_index(name='registros')
              .merge(df_cat_tienda, on='id_tienda')
              .sort_values('registros', ascending=False))
print(por_tienda.to_string(index=False))

 id_tienda  registros          nombre
        61      71612           Steam
         6      54498       Fanatical
        35      52772             GOG
        20      49576      GameBillet
        68      29151      Gamer Thor
        37      26320    Humble Store
        24      24865      GamersGate
        29      21140  GamesPlanet US
         2      20574      AllYouPlay
        15      20390       Dreamgame
        26      19336  GamesPlanet UK
        28      19277  GamesPlanet FR
        27      19104  GamesPlanet DE
        64      17937    WinGameStore
        42      17354 IndieGala Store
        49      15543          Newegg
        13      13009         DLGamer
        66      11040          Noctre
        73      10929      PlanetPlay
        50      10739          Nuuvem
        16      10717 Epic Game Store
